# Nuskha Pharmacies data warehouse: full pipeline

CT-472 Data Warehousing and BI project. **All data is synthetic (seed 472); no real patients, brands or companies.**

Each cell below calls code in `src/`; the notebook holds no logic of its own. Steps print `[OK]`, `[SKIP]` or `[FAILED]` with the last lines of their output.

## Setup
In Google Colab this installs and starts MySQL and PostgreSQL and creates the users (`scripts/setup_env.sh colab`). On a local machine the database services already run, so it only checks the five connections.

In [ ]:
import os, subprocess, sys
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ
if IN_COLAB:
    if not os.path.exists('src'):
        raise SystemExit('Upload or clone the project folder and %cd into it first.')
    subprocess.run(['bash', 'scripts/setup_env.sh', 'colab'], check=True)
subprocess.run([sys.executable, 'scripts/check_env.py', '--create'], check=True)

## Mode
`FULL_REBUILD` empties the warehouse and runs everything from the sources; `CONTINUE` keeps what exists and only adds what is missing (safe to repeat). The headless test run sets it with the environment variable `NUSKHA_MODE`.

In [ ]:
import os
MODE = os.environ.get('NUSKHA_MODE', 'CONTINUE')   # or 'FULL_REBUILD'
from src import pipeline
if MODE == 'FULL_REBUILD':
    pipeline.reset()
print('mode:', MODE)

## 1. Source systems
Four synthetic department databases are generated from seed 472 (FULL_REBUILD only) and loaded into MySQL (Sales, Finance), PostgreSQL (Purchasing) and SQLite (Inventory). The load is done twice and the per-table checksums are compared.

**Generator: reference data + stage-1 sources (Nov 2023 - Aug 2026)** — `python -m src.generate_sources --stage 1`

In [ ]:
pipeline.run_step('generate_stage_1', MODE)

**Generator: stage-2 sources (Sep - Oct 2026, corrections, deletions)** — `python -m src.generate_sources --stage 2`

In [ ]:
pipeline.run_step('generate_stage_2', MODE)

**Warehouse DDL: schemas dw, stg, etl, meta, ai** — `python -m src.create_warehouse`

In [ ]:
pipeline.run_step('create_warehouse', MODE)

**Load stage 1 into MySQL, PostgreSQL, SQLite (twice, checksums compared)** — `python -m src.load_sources --stage 1 --twice`

In [ ]:
pipeline.run_step('load_sources_stage_1', MODE)

## 2. Batch 1: extract, clean, match
Transaction tables are extracted by `updated_at` watermark into an untouched raw snapshot. Staging normalizes dates, units, money and currency; crosswalks give one product, branch and customer identity; bad rows go to `etl.rejects` with a reason code; rows and amounts are reconciled.

**Extract batch 1 by watermark into a raw snapshot** — `python -m src.extract --to 2026-08-31 23:59:59 --double-export data/generated/stage1/double_export.csv`

In [ ]:
pipeline.run_step('extract_batch_1', MODE)

**ETL batch 1: profile before** — `python -m src.etl_db profile-before --batch 1`

In [ ]:
pipeline.run_step('etl_profile-before_1', MODE)

**ETL batch 1: stage** — `python -m src.etl_db stage --batch 1`

In [ ]:
pipeline.run_step('etl_stage_1', MODE)

**ETL batch 1: profile after** — `python -m src.etl_db profile-after --batch 1`

In [ ]:
pipeline.run_step('etl_profile-after_1', MODE)

**ETL batch 1: crosswalk** — `python -m src.etl_db crosswalk --batch 1`

In [ ]:
pipeline.run_step('etl_crosswalk_1', MODE)

**ETL batch 1: prejoin** — `python -m src.etl_db prejoin --batch 1`

In [ ]:
pipeline.run_step('etl_prejoin_1', MODE)

**ETL batch 1: quality** — `python -m src.etl_db quality --batch 1`

In [ ]:
pipeline.run_step('etl_quality_1', MODE)

**ETL batch 1: reconcile** — `python -m src.etl_db reconcile --batch 1`

In [ ]:
pipeline.run_step('etl_reconcile_1', MODE)

## 3. Batch 1: warehouse load
Dimensions, then facts, as upserts on natural keys in one transaction per fact table, with lineage.

**Load batch 1 into the stars (dimensions, facts, lineage)** — `python -m src.load_db --batch 1`

In [ ]:
pipeline.run_step('load_batch_1', MODE)

**Dimension checks** — `python -m src.dw_checks dims --batch 1`

In [ ]:
pipeline.run_step('check_dims', MODE)

**Fact checks (rows, amounts, Diabetes Care margin)** — `python -m src.dw_checks facts --batch 1`

In [ ]:
pipeline.run_step('check_facts', MODE)

## 4. Batches, SCD Type 1 and failure handling
An unchanged rerun changes nothing; batch 2 brings late receipts, corrections, cancellations and deletions; the cough-syrup category fix is an SCD Type 1 overwrite; a forced failure rolls back.

**Rerun batch 1 unchanged: 0 rows change** — `python -m src.batch_tests rerun`

In [ ]:
pipeline.run_step('rerun_test', MODE)

**Stage-2 sources, extract batch 2, ETL, load, deletion check** — `python -m src.batch_tests batch2`

In [ ]:
pipeline.run_step('batch_2', MODE)

**SCD Type 1 category totals before / after** — `python -m src.batch_tests scd1`

In [ ]:
pipeline.run_step('scd1', MODE)

**FAIL_AT=0.5 rollback and clean rerun** — `python -m src.batch_tests failure`

In [ ]:
pipeline.run_step('controlled_failure', MODE)

## 5. Metadata, lineage and traces
The metadata repository, ERDs reverse-engineered from each engine's catalog, and six traces from a warehouse figure back to a source row.

**Metadata repository and lineage coverage** — `python -m src.metadata --load`

In [ ]:
pipeline.run_step('metadata', MODE)

**Reverse engineering: MySQL, PostgreSQL, SQLite** — `python -m src.reverse_engineer`

In [ ]:
pipeline.run_step('reverse_engineer', MODE)

## 6. Analytics and AI
Six saved views, twelve analytic queries, the sales forecast and anomaly detection (run twice to show the retry adds no rows).

**Saved views and their total checks** — `python -m src.views`

In [ ]:
pipeline.run_step('views', MODE)

**Six traces to a source row** — `python -m src.traces`

In [ ]:
pipeline.run_step('traces', MODE)

**Analytic queries A1-A12 with charts and readings** — `python -m src.queries`

In [ ]:
pipeline.run_step('queries', MODE)

**Forecast and anomaly detection, run twice (retry)** — `python -m src.ai_run --retry`

In [ ]:
pipeline.run_step('ai', MODE)

## 7. Validation and Power BI package
The 11-check validation suite, then the CSV exports, semantic model and report for Power BI.

**Validation suite (11 checks)** — `python -m src.validation`

In [ ]:
pipeline.run_step('validation', MODE)

**Power BI CSVs and reconciliation** — `python -m src.powerbi_export`

In [ ]:
pipeline.run_step('powerbi_export', MODE)

**Power BI semantic model (TMDL)** — `python -m src.powerbi_model`

In [ ]:
pipeline.run_step('powerbi_model', MODE)

**Power BI report (PBIR), schema-validated** — `python -m src.powerbi_report --validate`

In [ ]:
pipeline.run_step('powerbi_report', MODE)

## Final warehouse totals
These must be identical between runs and between this machine and Colab.

In [ ]:
t = pipeline.totals()
out = os.environ.get('NUSKHA_TOTALS_FILE')
if out:
    t.to_csv(out, index=False, lineterminator='\n')
t